<a href="https://colab.research.google.com/github/Alissa-Ouspen/data201_alissa/blob/main/week4/wk4_log_reg_assign_Alissa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alissa  -  Assignment – Week 4  
## Logistic Regression: R → Python Bridge

**Course:** Data Science  
**Submission:** Jupyter Notebook (.ipynb)  
**Dataset:** `housing.csv`

---

# Learning Objectives

By the end of this assignment, you should be able to:

- Fit and interpret **logistic regression models**
- Translate workflows from **R (glm / tidymodels)** to **Python**
- Interpret **odds ratios**
- Evaluate classification models using **Accuracy and ROC–AUC**
- Reflect on the difference between **statistical inference vs prediction**

---

# Dataset Description

The dataset contains **600 housing listings**.

| Variable | Description |
|---|---|
| listing_id | Unique identifier |
| price | Sale price of the house |
| size | House size (square footage) |
| bedrooms | Number of bedrooms |
| neighborhood | Location category |
| type | Housing type (SingleFamily, Townhouse, MultiFamily) |

---

# Step 0 – Create a Binary Outcome

For classification, convert price into a binary variable.

```
high_price = price > median(price)
```

This creates two groups:

- `1` → expensive homes  
- `0` → less expensive homes  

---



In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay, RocCurveDisplay

In [4]:
# Load the dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/Reben80/Data201/refs/heads/main/Dataset/housing.csv")

# Create the boolean outcome high_price
df["high_price"] = np.where(df["price"] > df["price"].median(), 1, 0)
# = np.where(df["price"] > np.median(df["price"]), 1, 0)

print(f"Median= {np.median(df["price"])}")
df.head()  # print(df["price"].median())

Median= 201165.0


,listing_id,price,size,bedrooms,neighborhood,type,high_price
0,100001,145143.0,1280.741760,1.0,Suburb,Townhouse,0
1,100002,152251.0,1406.283113,2.0,Uptown,SingleFamily,0
2,100003,148251.0,4146.825713,6.0,Suburb,MultiFamily,0
3,100004,177711.0,3946.599818,6.0,Suburb,SingleFamily,0
4,100005,155269.0,1243.751760,1.0,Downtown,MultiFamily,0


---

# Part A – Logistic Regression for Inference


### Report

Create a table including:

- coefficients
- odds ratios
- p-values

In [6]:
# Fit a logistic regression using statsmodels
model_sm = smf.logit(
    "high_price ~ size + bedrooms + C(neighborhood)",
    data = df
).fit()

print(model_sm.summary())

Optimization terminated successfully.
         Current function value: 0.684990
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:             high_price   No. Observations:                  524
Model:                          Logit   Df Residuals:                      517
Method:                           MLE   Df Model:                            6
Date:                Sun, 15 Mar 2026   Pseudo R-squ.:                 0.01177
Time:                        01:26:42   Log-Likelihood:                -358.93
converged:                       True   LL-Null:                       -363.21
Covariance Type:            nonrobust   LLR p-value:                    0.2006
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                         0.3796      0.289      1.311      0.

In [7]:
odds_ratios = model_sm.params.apply(np.exp)
odds_ratios

,0
Intercept,1.461647
C(neighborhood)[T.Midtown],0.736015
C(neighborhood)[T.Suburb],1.113030
C(neighborhood)[T.Uptown],1.048173
C(neighborhood)[T.Waterfront],1.511416
size,0.999479
bedrooms,1.261433


### Short Analysis
The size of the house is significantly (p = 0.037) correlated with whether the house is categorized as "expensive".  
This is significant at the α = 0.05 level.  
However, the Pearson's correlation coefficient for the "size" variable is -0.005, which means there is no linear relationship between size and price.  
There could be a significant but non-linear relationships between the variables which the logistic regression does not test for.  Since the sample size is very large (524), this could help explain the significance of that relationship.    

The Waterfront neighborhood has the higher odds ratio of expensive homes (1.51).  
This means the odds that an expensive home is on the Waterfront are approximately 0.6, while the odds it is in Downtown (the reference variable) is 0.4.  So, the odds that an expensive home is on the Waterfront is 1.5 times higher than the odds it is in Downtown.  

    
odds ratio =  $e^\beta$  
  

log (odds ratio) = $\beta_1$  = $log\frac{odds(Y=1|x=1)}{odds(Y=1|x=0)}$  
  
  
log odds = logit =  $log(\frac{p}{1-p})$ = $log\frac{odds(Y=1)}{odds(Y=0)}$

-----------------    

# Part B – Interpretation

For every additional bedroom, the odds of a house being expensive increase by 0.2614, or 26.14%.

Odds(size) = 0.9995.  Increase of .9995 = decrease of .0005.  

For every additional square foot of size, the odds of a house being expensive decrease by 0.0005, or 0.05%.



---


  


# Part C – Prediction Workflow  
## (tidymodels → scikit-learn)

### 1. Train/Test Split

Split the data:

- 80% training
- 20% testing